# Bab 5 - Latihan 1: Asisten Serah Terima Shift dengan Memori Persisten

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dteti-workshop/workshop-genai-migas/blob/main/colab/hari3/05-memori-agen-latihan-1.ipynb)

Ini versi **Google Colab**. Isi dan urutan contohnya sama dengan versi VS Code; yang berbeda hanya cell Persiapan di bawah, yang menyalin repository, memasang pustaka, dan mengisi API key di runtime Colab.

1. Pilih **File > Save a copy in Drive** agar perubahan tersimpan di akun Anda sendiri.
2. Jalankan tiga cell **Persiapan Colab** berurutan (sekitar 2-3 menit pada sesi baru).
3. Kerjakan dari atas ke bawah dengan `Shift+Enter`.

Kode jawaban latihan ada di bagian **Kode Jawaban Latihan** pada [PDF bab ini](https://github.com/dteti-workshop/workshop-genai-migas/blob/main/hari3/05-memori-agen/05-memori-agen.pdf). Salin kodenya ke cell yang bertanda `TODO`.

Kalau runtime terputus, file di `/content` (termasuk `.env`) hilang. Jalankan ulang Persiapan, lalu lanjutkan dari cell yang sedang Anda kerjakan.

## Persiapan Colab
Jalankan tiga cell berikut berurutan sebelum cell lain di notebook ini.

In [ ]:
# Persiapan 1 - Menyalin repository dan memasang pustaka (sekitar 2-3 menit pada sesi baru)
import os
import subprocess
import sys

REPO_URL = "https://github.com/dteti-workshop/workshop-genai-migas.git"
REPO = "/content/workshop-genai-migas" if os.path.isdir("/content") else os.path.abspath("workshop-genai-migas")
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
os.chdir(REPO)

pasang = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "colab/requirements-colab.txt", "-e", "."],
                        capture_output=True, text=True)
if pasang.returncode != 0:
    print(pasang.stderr[-3000:])
    raise RuntimeError("Instalasi pustaka gagal. Jalankan ulang cell ini.")

# "-e ." memasang common dan migas_shared agar subproses (server MCP) dapat mengimpornya; kernel yang sedang berjalan
# belum memuat instalasi baru, jadi folder repo juga ditambahkan ke sys.path
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.environ["PYTHONPATH"] = REPO + os.pathsep + os.environ.get("PYTHONPATH", "")
print("Repository dan pustaka siap. Folder kerja:", os.getcwd())

In [ ]:
# Persiapan 2 - Mengisi ID peserta dan API key (dilewati bila .env sudah berisi key)
import re
from getpass import getpass
from pathlib import Path

from dotenv import load_dotenv

env = Path(REPO) / ".env"


def rahasia(nama: str):
    """Nilai dari Colab Secrets (ikon kunci di panel kiri), atau None bila tidak ada."""
    try:
        from google.colab import userdata
        return userdata.get(nama)
    except Exception:  # noqa: BLE001 - bukan di Colab, secret belum dibuat, atau akses notebook belum diizinkan
        return None


if not (env.exists() and re.search(r"(?m)^ONLINE_API_KEY=(?!isi-dengan)\S+", env.read_text(encoding="utf-8"))):
    peserta = (rahasia("PARTICIPANT_ID") or input("ID peserta (nomor meja dan nama singkat, contoh P07-andi): ")).strip()
    kunci = (rahasia("ONLINE_API_KEY") or getpass("API key workshop (ketikan tidak ditampilkan): ")).strip()
    if not kunci:
        raise ValueError("API key belum diisi. Jalankan ulang cell ini.")
    isi = (Path(REPO) / ".env.example").read_text(encoding="utf-8")
    for nama, nilai in (("PARTICIPANT_ID", peserta or "P00-nama"), ("ONLINE_API_KEY", kunci)):
        isi = re.sub(rf"(?m)^{nama}=.*$", lambda _, n=nama, v=nilai: f"{n}={v}", isi)
    env.write_text(isi, encoding="utf-8")
    del kunci

# Nilai dari .env dimuat ulang; modul yang sudah diimpor dibuang agar membaca nilai terbaru
load_dotenv(env, override=True)
for modul in [m for m in sys.modules if m.split(".")[0] in ("common", "migas_shared", "rag_pipeline")]:
    del sys.modules[modul]
print("File .env siap untuk peserta", os.environ.get("PARTICIPANT_ID"))

In [ ]:
# Persiapan 3 - Memeriksa koneksi ke model
from migas_shared.llm import chat

try:
    print("Jawaban model:", chat("Balas hanya dengan kata: SIAP").strip())
except Exception as error:  # noqa: BLE001
    raise RuntimeError("Model tidak menjawab. Periksa API key (jalankan ulang Persiapan 2) dan koneksi internet. "
                       f"Pesan asli: {error}") from error

Isi bagian bertanda `TODO(...)` dengan kode dari bagian **Kode Jawaban Latihan** pada PDF bab ini, lalu jalankan sel satu per satu
dengan Shift+Enter. Baris `... = None  # TODO` atau `raise NotImplementedError` berarti kode itu harus Anda ganti.

Asisten yang mengingat setiap operator, bahkan setelah program dimulai ulang.
- Memori disimpan di file SQLite (SqliteSaver)
- Setiap nama operator memiliki thread sendiri
- Perintah pada mode interaktif: /riwayat, /baru, /keluar

Bagian utama berupa demo berskrip yang menirukan: pakai aplikasi, tutup, lalu buka lagi.
Sel terakhir membuka mode interaktif (ketik pesan sendiri) setelah `INTERAKTIF` diubah menjadi `True`.

In [ ]:
# Langkah 1 - Impor dan fungsi bantu
import os
import sqlite3
import tempfile
import uuid

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.sqlite import SqliteSaver

from migas_shared.config import ROOT

load_dotenv()

DB_BAWAAN = str(ROOT / "hari3" / "05-memori-agen" / "memori_chat.db")


def buat_agen(db_path: str):
    model = ChatOpenAI(
        model=os.getenv("ONLINE_MODEL"),
        base_url=os.getenv("ONLINE_BASE_URL"),
        api_key=os.getenv("ONLINE_API_KEY"),
        reasoning_effort=os.getenv("LLM_REASONING_EFFORT", "none"),
    )
    # TODO(B5.1): buka koneksi SQLite (check_same_thread=False), buat agen tanpa tool dengan checkpointer=SqliteSaver(connection), lalu kembalikan (agent, connection)
    raise NotImplementedError("TODO(B5.1)")


def chat(agent, user: str, sesi: str, teks: str) -> str:
    """Satu giliran percakapan. user dan sesi bersama-sama menentukan thread."""
    # TODO(B5.2): buat config dengan thread_id '<user>:<sesi>', panggil agent.invoke, dan kembalikan teks jawaban terakhir
    raise NotImplementedError("TODO(B5.2)")


def jumlah_pesan(agent, user: str, sesi: str) -> int:
    # TODO(B5.3): kembalikan jumlah pesan pada state thread itu lewat agent.get_state(config)
    raise NotImplementedError("TODO(B5.3)")


# Folder sementara untuk file database demo; dibersihkan di sel terakhir demo
folder = tempfile.TemporaryDirectory()
db_path = os.path.join(folder.name, "demo.db")

## Langkah 2 - Jalan 1: pakai aplikasi
Rina memperkenalkan diri dan menyebut jam tersibuknya. Percakapan disimpan di file SQLite.

In [ ]:
print("Jalan 1")
agent, connection = buat_agen(db_path)
for teks in ["Saya Rina, supervisor shift malam di GS-SR1.", "Jam tersibuk kami adalah pukul 07.00."]:
    print(f"rina: {teks}")
    print(f"Agen: {chat(agent, 'rina', 'utama', teks)}")
print(f"Pesan tersimpan untuk rina: {jumlah_pesan(agent, 'rina', 'utama')}")
connection.close()

## Langkah 3 - Jalan 2: program dimulai ulang
Agen dan koneksi baru membaca file yang sama. Riwayat Rina dimuat, sedangkan Agus belum punya riwayat.

In [ ]:
print("Jalan 2 (program dimulai ulang)")
agent, connection = buat_agen(db_path)
print(f"Pesan dimuat untuk rina: {jumlah_pesan(agent, 'rina', 'utama')}")
print(f"Pesan dimuat untuk agus: {jumlah_pesan(agent, 'agus', 'utama')}")
pertanyaan = "Di gathering station mana saya bertugas dan kapan jam tersibuk saya?"
print(f"rina: {pertanyaan}")
print(f"Agen: {chat(agent, 'rina', 'utama', pertanyaan)}")

## Langkah 4 - Sesi baru untuk Rina
Sesi baru berarti thread baru, jadi agen tidak mengingat percakapan sebelumnya.

In [ ]:
print("Sesi baru untuk rina (thread baru)")
print(f"rina: {pertanyaan}")
print(f"Agen: {chat(agent, 'rina', 'sesi2', pertanyaan)}")
connection.close()
folder.cleanup()

## Langkah 5 - Mode interaktif (opsional)
Ubah `INTERAKTIF` menjadi `True`, jalankan sel, lalu ketik nama dan pesan Anda. Memori disimpan di
`hari3/05-memori-agen/memori_chat.db`, jadi percakapan berlanjut saat sel dijalankan lagi.
Perintah: `/riwayat` (jumlah pesan), `/baru` (sesi baru), `/keluar` (berhenti).

In [ ]:
INTERAKTIF = False  # ubah ke True untuk mencoba sendiri

if INTERAKTIF:
    user = input("Nama Anda: ").strip().lower() or "tamu"
    sesi = "utama"
    agent, connection = buat_agen(DB_BAWAAN)

    ada = jumlah_pesan(agent, user, sesi)
    if ada:
        print(f"Selamat datang kembali, {user}! {ada} pesan tersimpan dimuat.")
    print("Perintah: /riwayat  /baru  /keluar\n")

    while True:
        teks = input("Anda: ").strip()
        if teks == "/keluar":
            break
        if teks == "/riwayat":
            print(f"   {jumlah_pesan(agent, user, sesi)} pesan dalam sesi ini")
        elif teks == "/baru":
            sesi = uuid.uuid4().hex[:6]
            print(f"   Sesi baru dimulai ({sesi}). Sesi lama tetap tersimpan.")
        elif teks:
            print(f"Agen: {chat(agent, user, sesi, teks)}\n")
    connection.close()